# demo_inference.ipynb

Batch inference notebook for running predictions over a folder of sample images.

## Notes

- Run batch inference over a folder of images.
- Useful for creating screenshots for the GitHub README.

In [20]:
from pathlib import Path
import tensorflow as tf
from predict import load_and_prepare_image, CLASS_NAMES
import numpy as np

args = type('Args', (), {
    'input_path': Path("test_images"),  # folder or single file
    'model': Path("../model/fruit_classifier_final_best_model.keras"),
    'img_size': 64                    # ← must match model input
})()

supported_exts = {".jpg", ".jpeg", ".png", ".webp", ".avif"}

print(f"Input path: {args.input_path.resolve()}")
print(f"Model path: {args.model.resolve()}")

if not args.input_path.exists():
    print(f"Input path not found: {args.input_path}")
elif not args.model.exists():
    print(f"Model not found: {args.model}")
else:
    model = tf.keras.models.load_model(args.model)

    if args.input_path.is_file():
        image_paths = [args.input_path] if args.input_path.suffix.lower() in supported_exts else []
    elif args.input_path.is_dir():
        image_paths = [
            p for p in sorted(args.input_path.iterdir())
            if p.is_file() and p.suffix.lower() in supported_exts
        ]
    else:
        image_paths = []

    if not image_paths:
        print("No supported image files found.")
    else:
        for image_path in image_paths:
            try:
                x = load_and_prepare_image(image_path, args.img_size)
                probs = model.predict(x, verbose=0)[0]
                pred_idx = int(np.argmax(probs))
                print(f"{image_path.name} -> {CLASS_NAMES[pred_idx]} ({float(probs[pred_idx]):.4f})")
            except Exception as e:
                print(f"Skipped {image_path.name}: {e}")

Input path: C:\Users\Candy\OneDrive\IT\CandyCheng-git\AI-ML-CNN-Fruit-Classification\src\test_images
Model path: C:\Users\Candy\OneDrive\IT\CandyCheng-git\AI-ML-CNN-Fruit-Classification\model\fruit_classifier_final_best_model.keras
Avocado.png -> Watermelon (0.6299)
pineapple.png -> Avocado (0.8519)
